# Gold Layer

Imports


In [0]:
from pyspark.sql import functions as F

In [0]:
FACILITY_SILVER_TABLE = "databricks_project1.silver.facility"
FACILITY_GOLD_TABLE = "databricks_project1.gold.dim_facility"

Read Silver


In [0]:
facility_silver_df = spark.table(FACILITY_SILVER_TABLE)

print("Silver row count:", facility_silver_df.count())

display(facility_silver_df)

Transformes

In [0]:
facility_gold_df = (
    facility_silver_df
    .select(
        F.col("Facility_Code").cast("int").alias("Facility_Code"),
        F.trim(F.col("Facility_Name")).alias("Facility_Name")
    )
    .dropDuplicates(["Facility_Code"])
)

print("Gold row count:", facility_gold_df.count())

display(facility_gold_df)

Data quality checks

In [0]:
print("===== DATA QUALITY CHECK =====")

# Null Facility Code
null_code_count = facility_gold_df.filter(
    F.col("Facility_Code").isNull()
).count()

# Null Facility Name
null_name_count = facility_gold_df.filter(
    F.col("Facility_Name").isNull()
).count()

# Duplicate Facility Code
duplicate_code_count = (
    facility_gold_df
    .groupBy("Facility_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Null Facility Codes :", null_code_count)
print("Null Facility Names :", null_name_count)
print("Duplicate Codes     :", duplicate_code_count)

Write to gold

In [0]:
(
    facility_gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FACILITY_GOLD_TABLE)
)

print("Dim_Facility successfully created")

Validations

In [0]:
final_facility_df = spark.table(FACILITY_GOLD_TABLE)

print("Final Dim_Facility count:", final_facility_df.count())

final_facility_df.printSchema()

display(final_facility_df)